# Human-in-the-loop tool confirmation

GeoAgent classifies tools as either *safe* (read-only, navigation, preview) or *confirmation-required* (destructive, expensive, externally-visible). When a confirmation-required tool is about to fire, the deepagents graph pauses and emits an `__interrupt__`. `GeoAgent.chat()` routes that interrupt through a `ConfirmCallback` you supply at construction.

This notebook walks through the three built-in callbacks plus a custom one.


## Setup

In [ ]:
from geoagent import (
    GeoAgent,
    ConfirmCallback,
    ConfirmRequest,
    auto_approve_all,
    auto_approve_safe_only,
    geo_tool,
)

## Define a confirmation-required tool

We define a tiny toy tool so the demo does not depend on QGIS, Earth Engine, or a live map. Setting `requires_confirmation=True` flips it into the interrupt path.

In [ ]:
@geo_tool(
    category="data",
    description="Pretend to delete a dataset by name.",
    requires_confirmation=True,
)
def delete_dataset(name: str) -> str:
    """Pretend to delete the named dataset (no-op for the demo)."""
    return f"deleted dataset: {name}"


delete_dataset.metadata

## Default callback rejects everything

When no `confirm=` is supplied, `GeoAgent` uses `auto_approve_safe_only`, which **rejects** every confirmation-required call. Safe tools still run, but the destructive `delete_dataset` cannot fire silently.

In [ ]:
agent = GeoAgent(tools=[delete_dataset])
result = agent.chat("Delete the dataset named lulc-2024.")
print("success:", result.success)
print("executed:", result.executed_tools)
print("cancelled:", result.cancelled_tools)

## A custom callback that prints the request and approves

A `ConfirmCallback` is just `Callable[[ConfirmRequest], bool]`. The request includes the proposed tool name, the args, the description (lifted from the `@geo_tool` docstring), the category, and the full GeoAgent metadata dict.

Notebooks cannot reliably block on `input()`, so this demo callback prints the request and returns `True`. Replace the body with `input('approve? ') == 'y'` in a CLI session to gate on real user input.

In [ ]:
def interactive_confirm(req: ConfirmRequest) -> bool:
    print("Confirm request:")
    print(f"  tool: {req.tool_name}")
    print(f"  args: {req.args}")
    print(f"  description: {req.description}")
    print(f"  category: {req.category}")
    print(f"  metadata: {req.metadata}")
    return True


agent = GeoAgent(tools=[delete_dataset], confirm=interactive_confirm)
result = agent.chat("Delete the dataset named lulc-2024.")
print("---")
print("executed:", result.executed_tools)
print("cancelled:", result.cancelled_tools)
print("answer:", result.answer_text)

## `auto_approve_all` — only safe in fully sandboxed sessions

`auto_approve_all` is provided for tests and trusted automation. Every confirmation request returns `True`, so destructive tools fire without human review. Do not use this as a default in interactive applications.

In [ ]:
agent = GeoAgent(tools=[delete_dataset], confirm=auto_approve_all)
result = agent.chat("Delete the dataset named lulc-2024.")
print("executed:", result.executed_tools)
print("cancelled:", result.cancelled_tools)

## How it wires together

1. `@geo_tool(requires_confirmation=True)` stamps GeoAgent metadata onto the   `BaseTool`.
2. The factory at construction time builds a deepagents `interrupt_on` from   every confirmation-required tool's name.
3. When the agent proposes one of those tools, the graph emits   `__interrupt__` with `action_requests`.
4. `GeoAgent.chat()` routes each pending request through your   `ConfirmCallback`, then resumes the graph with   `Command(resume={'decisions': [...]})`.
5. Approved tools run; rejected tools land in `result.cancelled_tools` and   their stub `ToolMessage`s are excluded from `result.executed_tools`.
